# N061 · 堆调度与离散事件模拟

**目标：** 区分按时间推进与逐时刻暴力模拟。

**先修：** N036, N057, N059。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 就绪队列；结束事件；空闲时间跳转；排序复合键；双堆。

**配套讲解来源：** [同名逐章意见](../../comment/061_heap_event_scheduling.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章用堆来模拟"任务排队使用机器"的调度过程，解决两道题：

1. **单线程 CPU**：n 个任务，第 i 个在 `tasks[i][0]` 时刻到达、需要执行 `tasks[i][1]` 那么长的时间。只有一台 CPU，它在每个时刻要么闲着（没有已到达的任务），要么从"已到达且未执行"的任务里挑一个执行到完（不可抢占）。挑选规则：先挑执行时间最短的；时间相同挑原下标最小的。问任务被完成的先后顺序。
2. **服务器分配**：m 台服务器各有权重 `servers[i]`，第 j 个任务在时刻 j 到达、需要 `tasks[j]` 长的时间。每台服务器同一时刻只能干一个活，忙完才空闲。分配规则：从"此刻空闲"的服务器里挑权重最小的；权重相同挑下标最小的；如果全部服务器都在忙，就等到最早忙完的那台空出来。问每个任务分给哪台服务器。

具体到数字的小例子：

- 单线程 CPU：任务 `[[1,2],[2,4],[3,2],[4,1]]`（到达、时长），输出 `[0,2,3,1]`。为什么？时刻 1 只有任务 0 到达，先做它（做到 3）；此时任务 1（时刻 2 到达）和任务 2（时刻 3 到达）都在等，任务 2 时长 2 比任务 1 的 4 短，先做任务 2（做到 5）；此刻任务 3 也到了，堆里剩任务 1（时长 4）和任务 3（时长 1），先做任务 3（做到 6）；最后做任务 1。完成顺序 0,2,3,1。
- 服务器：权重 `[3,3,2]`，任务时长 `[1,2,3,2,1,2]`，输出 `[2,2,0,2,1,2]`。为什么？时刻 0 三台全空，权重最小的是 2 号（权重 2），给它；它 1 时刻做完，时刻 1 的任务又给它；时刻 2 只有 0 号、1 号空着，权重并列取下标小的 0 号；以此类推。

两题共同的关键想法是：**不要一秒一秒地模拟时间**。两个事件之间（比如 CPU 空闲等任务到达、服务器全忙等释放）可能隔着巨大的时间空档，直接把时钟"跳"到下一个事件时刻就行。

## 2. 基础知识：先读懂这些词

- **离散事件模拟**：时间不是连续 tick 的，而是被"事件"（任务到达、任务完成、服务器释放）切成分离的时刻。模拟只在事件点上推进，两个事件之间什么都不会变，跳过去不吃亏。
- **就绪队列（ready）**：单线程 CPU 里"已到达、等着被执行"的任务堆。键是 `(时长, 原下标)`，正好一条键同时实现"短作业优先 + 下标破平局"两条规则。
- **结束/释放事件（busy 堆）**：服务器题里记录"哪台服务器什么时候忙完"。键是 `(释放时刻, 权重, 下标)`——释放时刻放第一位，因为"谁先空出来"最先被查。
- **空闲时间跳转**：CPU 没活干时，`now=max(now, 下一个到达时刻)`；服务器全忙时，`now=max(now, 最早释放时刻)`。一行 max 就代替了几百万次逐秒空转。
- **排序复合键**：把多条比较规则压进一个元组，Python 比较元组时先比第一项、平了再比第二项。`(duration,index)`、`(weight,index)`、`(free_time,weight,index)` 都是这一招。
- **双堆（可用堆 + 忙碌堆）**：服务器题的核心结构——一台服务器任一时刻恰好在其中一个堆里：空闲的在 available（按权重排），干活的在 busy（按释放时刻排）。任务完成只是把节点从一个堆搬进另一个堆。

## 3. 思路推导：从小例子开始

Step 1：先把任务按到达时刻排序（单线程 CPU 里原下标必须随身携带，它是破平局的依据）。

Step 2：主循环——只要还有任务没入队或堆里还有活，就先看"现在有没有可做的活"：没有就把时钟跳到下一个到达时刻；有就把到达时刻不超过当前时钟的任务全部收进就绪堆。

Step 3：从就绪堆弹出堆顶执行（单线程：输出它的下标，时钟加上它的时长；时钟就这么被"执行"推着往前走）。

Step 4：服务器题改成"每个时刻 j 固定来一个任务"：先把 busy 堆里释放时刻不超过当前时钟的服务器全部搬回 available，再从 available 弹出 (权重, 下标) 最小的那台分配，并把 `(now+时长, 权重, 下标)` 压进 busy。

Step 5：全忙时先跳时钟：available 空且 busy 非空，就把时钟跳到 busy 堆顶的释放时刻，再回 Step 4 收割释放的服务器。

手算演示（单线程 CPU，notebook 小实例原数据）：任务 `[[1,2],[2,4],[3,2],[4,1]]`，按到达排序后是 `(1,2,#0), (2,4,#1), (3,2,#2), (4,1,#3)`。

| 执行次序 | 任务索引 | 到达 | 处理时长 | 这一步发生了什么 |
| --- | --- | --- | --- | --- |
| 1 | 0 | 1 | 2 | now 从 0 跳到 1（等待首个到达），执行到 3 |
| 2 | 2 | 3 | 2 | 时刻 3 收编任务 1、2 入就绪堆，(2,#2) < (4,#1) 先执行，到 5 |
| 3 | 3 | 4 | 1 | 时刻 4 到达的任务 3 已入堆，(1,#3) 最短，执行到 6 |
| 4 | 1 | 2 | 4 | 只剩任务 1，执行到 10，输出顺序 [0,2,3,1] |

这张表与 notebook 小实例生成的表格一致。另一个用例 `[[100,2],[1000,1]]` 更能体现"跳时间"：CPU 做完任务 0 后时刻是 102，下一个任务 1000 时刻才到，中间 898 个时间单位一秒都不用模拟，`now` 直接跳到 1000。

再手算一遍服务器分配（notebook 小实例的第二张表原数据）：权重 `[3,3,2]`、任务时长 `[1,2,3,2,1,2]`，任务的到达时刻就是它的下标 0~5。

| 任务到达时刻 | 分配服务器 | 这一步发生了什么 |
| --- | --- | --- |
| 0 | 2 | 三台全空，2 号权重 2 最小，忙到 1 |
| 1 | 2 | 2 号 1 时刻释放正好赶上，继续用它，忙到 3 |
| 2 | 0 | 2 号忙到 3 未释放；0 号、1 号空闲且权重并列，取下标小的 0 号，忙到 5 |
| 3 | 2 | 3 时刻 2 号释放，重新入选，权重仍最小，忙到 5 |
| 4 | 1 | 两台忙到 5 的都未释放，只剩 1 号空闲，忙到 5 |
| 5 | 2 | 三台在 5 时刻同时释放（按释放时刻、权重、下标先后搬回），再选 2 号 |

六次分配连起来正是输出 `[2,2,0,2,1,2]`，与 notebook 小实例生成的"任务到达时刻—分配服务器"表格一致。时刻 5 那一行尤其值得看：三台服务器同时到期，busy 堆按 `(释放时刻, 权重, 下标)` 排序保证它们被全部搬回 available 后，才进行这一次分配。

## 4. 核心代码：single_threaded_order

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def single_threaded_order(tasks):
    pending = sorted(((start, duration, i) for i, (start, duration) in enumerate(tasks)))
    ready = []
    i = 0
    now = 0
    out = []
    while i < len(pending) or ready:
        if not ready:
            now = max(now, pending[i][0])
        while i < len(pending) and pending[i][0] <= now:
            start, duration, index = pending[i]
            heapq.heappush(ready, (duration, index))
            i += 1
        duration, index = heapq.heappop(ready)
        out.append(index)
        now += duration
    return out
```

### 逐段读懂代码

### 1. `single_threaded_order`：单线程 CPU

```python
import heapq

def single_threaded_order(tasks):
    pending = sorted(((start, duration, i) for i, (start, duration) in enumerate(tasks)))
    ready = []
    i = 0
    now = 0
    out = []
    while i < len(pending) or ready:
        if not ready:
            now = max(now, pending[i][0])
        while i < len(pending) and pending[i][0] <= now:
            start, duration, index = pending[i]
            heapq.heappush(ready, (duration, index))
            i += 1
        duration, index = heapq.heappop(ready)
        out.append(index)
        now += duration
    return out
```

- `pending` 把任务变成 `(到达, 时长, 原下标)` 并排序，之后 `i` 是"下一个还没入队的任务"的指针。
- 主循环条件 `while i<len(pending) or ready`：还有没入队的任务、或者就绪堆里还有活，二者任一成立模拟就继续。
- `if not ready: now=max(now,pending[i][0])` 是空闲跳转：堆空说明 CPU 闲着，`max` 保证时钟只进不退（执行完的任务可能已把 now 推得更晚）。注意此处 `pending[i]` 必存在——堆空且 i 越界的话循环条件早就为假了。
- 内层 while 把"到达时刻 <= now"的任务批量收编，压入的键是 `(duration,index)`，不携带 start（start 已经过期，再无用处）。
- 弹出堆顶、记录下标、`now+=duration`——"执行任务"就是把时钟推过它的时长。不可抢占体现在：弹出的任务一定执行到完，中途不回头。

### 2. `assign_tasks`：服务器分配

```python
def assign_tasks(servers, tasks):
    if not servers and tasks:
        raise ValueError('at least one server required')
    available = [(weight, i) for i, weight in enumerate(servers)]
    heapq.heapify(available)
    busy = []
    now = 0
    out = []
    for arrival, duration in enumerate(tasks):
        now = max(now, arrival)
        if not available and busy:
            now = max(now, busy[0][0])
        while busy and busy[0][0] <= now:
            _, weight, index = heapq.heappop(busy)
            heapq.heappush(available, (weight, index))
        weight, index = heapq.heappop(available)
        out.append(index)
        heapq.heappush(busy, (now + duration, weight, index))
    return out
```

- 第一行守卫：有任务却一台服务器都没有，永远无法分配，直接报错。
- `available` 一开始 O(m) 建堆装下全部空闲服务器，键 `(weight,i)` 同时编码"权重小优先 + 下标小破平局"。
- 任务到达时刻就是它的下标（`enumerate`），所以循环变量直接叫 arrival。
- `now=max(now,arrival)` 保证时钟不倒退（上一台服务器忙完的时刻可能已经超过当前任务的下标）。
- `if not available and busy: now=max(now,busy[0][0])`：全忙时把时钟跳到"最早释放时刻"。busy 按 `(释放时刻,权重,下标)` 排，堆顶就是最早空出来的那台。
- 内层 while 收割所有"释放时刻 <= now"的服务器，从 busy 搬回 available（`_` 丢掉已过期的释放时刻，搬回的是 `(weight,index)`）。
- 然后从 available 弹出最优服务器记录进 out，并把 `(now+duration, weight, index)` 压进 busy——它将忙到 now+duration。busy 元组里的 weight、index 主要是为了让元组规范可比，关键第一位是释放时刻。

## 5. 分段实现：按顺序运行

**本章接口：** `single_threaded_order(tasks)`、`assign_tasks(servers, tasks)`

### 导入本章使用的库

In [1]:
import heapq

### single_threaded_order

In [2]:
def single_threaded_order(tasks):
    pending = sorted(((start, duration, i) for i, (start, duration) in enumerate(tasks)))
    ready = []
    i = 0
    now = 0
    out = []
    while i < len(pending) or ready:
        if not ready:
            now = max(now, pending[i][0])
        while i < len(pending) and pending[i][0] <= now:
            start, duration, index = pending[i]
            heapq.heappush(ready, (duration, index))
            i += 1
        duration, index = heapq.heappop(ready)
        out.append(index)
        now += duration
    return out

### assign_tasks

In [3]:
def assign_tasks(servers, tasks):
    if not servers and tasks:
        raise ValueError('at least one server required')
    available = [(weight, i) for i, weight in enumerate(servers)]
    heapq.heapify(available)
    busy = []
    now = 0
    out = []
    for arrival, duration in enumerate(tasks):
        now = max(now, arrival)
        if not available and busy:
            now = max(now, busy[0][0])
        while busy and busy[0][0] <= now:
            _, weight, index = heapq.heappop(busy)
            heapq.heappush(available, (weight, index))
        weight, index = heapq.heappop(available)
        out.append(index)
        heapq.heappush(busy, (now + duration, weight, index))
    return out

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

tasks=[[1,2],[2,4],[3,2],[4,1]]; order=single_threaded_order(tasks)
show_table(['执行次序','任务索引','到达','处理时长'],[(j,i,*tasks[i]) for j,i in enumerate(order)])
show_table(['任务到达时刻','分配服务器'],list(enumerate(assign_tasks([3,3,2],[1,2,3,2,1,2]))))

执行次序,任务索引,到达,处理时长
0,0,1,2
1,2,3,2
2,3,4,1
3,1,2,4


任务到达时刻,分配服务器
0,2
1,2
2,0
3,2
4,1
5,2


## 7. 正确性、适用条件与复杂度

事件时刻之间没有候选集合变化，直接跳跃不会改变决策。每次分配前释放所有已到期资源，再用完整并列规则取堆顶，与逐资源扫描定义一致。

**代价：** CPU调度O(n log n)时间/O(n)空间；服务器调度O((n+m)log m)时间/O(m)工作空间，不计输出。

### 展开理解

为什么"跳时间"不会改变结果？因为在两个相邻事件之间，就绪集合一个成员都不增不减：没有新任务到达（否则那是下一个事件）、没有任务完成（同理），CPU 或服务器的可选集合完全静止，早决策晚决策拿到的候选一模一样。所以把空档整段跳过，等价于一秒一秒熬过去，只是不用真的熬。

为什么每次分配前"先释放所有到期服务器"就够了？因为释放事件只会让候选集合变大、不会改变已释放者的权重和下标；先把到期的全部搬回 available，再取堆顶，等价于把所有空闲服务器扫一遍取 (权重,下标) 最小的——与题目逐台扫描的定义一致。busy 堆按释放时刻排序又保证我们释放的恰好是"所有该释放的"，一个不漏。

复杂度：单线程 CPU 每个任务入队一次、出队一次，就绪堆操作 O(log n)，排序 O(n log n)，总计 O(n log n) 时间、O(n) 空间。n=10⁵ 时约几百万次比较，瞬间完成。服务器题：available 建堆 O(m)，每个任务触发至多几轮 busy→available 搬运，每台服务器每次搬运 O(log m)，n 个任务 m 台服务器合计 O((n+m) log m) 时间；工作空间 O(m)（两个堆合起来每个服务器任一时刻只占一个位置，不计输出的 out 列表）。举例 m=10⁵、n=10⁵：约 10⁶×17 量级的比较，一两秒内跑完。

## 8. 单元测试：每个用例在检查什么

```python
assert single_threaded_order([[1, 2], [2, 4], [3, 2], [4, 1]]) == [0, 2, 3, 1]
```
正常用例：就是第三节手算的表，覆盖"短作业优先"（任务 2 抢在更早到达的任务 1 前面）和"下标破平局"之外的常规路径。

```python
assert single_threaded_order([[100, 2], [1000, 1]]) == [0, 1]
```
大空档用例：两个任务相隔 900 个时间单位，专门测"空闲跳转"——若逐秒模拟会做 900 次无意义循环，跳转版一步跨过，输出顺序仍是先后到达的 0、1。

```python
assert assign_tasks([3, 3, 2], [1, 2, 3, 2, 1, 2]) == [2, 2, 0, 2, 1, 2]
```
正常用例：权重最小的 2 号服务器反复被选中（前两次），全忙跳转和释放收割的路径都被走到：时刻 2 只有 0/1 空闲（权重并列取 0），时刻 4 只剩 1 号，时刻 5 三台同时释放又回到 2 号。

```python
for _ in range(200):
    weights = [rng.randrange(1, 5) for _ in range(rng.randrange(1, 5))]
    tasks = [rng.randrange(1, 8) for _ in range(15)]
    free = [0] * len(weights)
    now = 0
    ref = []
    for arrival, duration in enumerate(tasks):
        now = max(now, arrival)
        if all((t > now for t in free)):
            now = min(free)
        i = min((i for i, t in enumerate(free) if t <= now), key=lambda i: (weights[i], i))
        ref.append(i)
        free[i] = now + duration
    assert assign_tasks(weights, tasks) == ref
```
随机对照测试：右侧手写了一个完全按题面定义的暴力参照——逐个任务检查每台服务器的空闲时刻 `free`，全忙就等到最早空闲，再按 (权重,下标) 挑。跑 200 组随机输入（1~4 台服务器、15 个任务、权重和时长都随机），两边结果逐位比对。这个参照故意写得"笨"，正因为它不聪明，才可信。

In [5]:
assert single_threaded_order([[1, 2], [2, 4], [3, 2], [4, 1]]) == [0, 2, 3, 1]

assert single_threaded_order([[100, 2], [1000, 1]]) == [0, 1]

assert assign_tasks([3, 3, 2], [1, 2, 3, 2, 1, 2]) == [2, 2, 0, 2, 1, 2]

from random import Random

rng = Random(61)

for _ in range(200):
    weights = [rng.randrange(1, 5) for _ in range(rng.randrange(1, 5))]
    tasks = [rng.randrange(1, 8) for _ in range(15)]
    free = [0] * len(weights)
    now = 0
    ref = []
    for arrival, duration in enumerate(tasks):
        now = max(now, arrival)
        if all((t > now for t in free)):
            now = min(free)
        i = min((i for i, t in enumerate(free) if t <= now), key=lambda i: (weights[i], i))
        ref.append(i)
        free[i] = now + duration
    assert assign_tasks(weights, tasks) == ref

print('N061: 所有本章断言通过')

N061: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 构造相同到达时间的多任务。

**练习 2：** 实现小规模逐时刻参照。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- 练习 1（构造相同到达时间的多任务）：让一堆任务的 start 全相同，比如 `[[5,3],[5,1],[5,2]]`。提示：这时就绪堆一次收编全部，堆序完全由 `(duration,index)` 决定，输出应当是时长 1、2、3 对应的下标顺序；再构造"时长也相同"的输入验证下标破平局。手算后用本章接口对拍。
- 练习 2（实现小规模逐时刻参照）：给 `single_threaded_order` 写一个"一秒一秒熬"的暴力版本：时钟每加 1，检查有没有新任务到达；CPU 空闲且堆非空就开干。提示：要点是"任务一旦开干要锁住直到完成"，用一个 `(剩余时长, 下标)` 变量表示正在执行的任务；小输入（比如时刻不超过几百）下随机对拍 100 组，体会两种写法结果一致但代价天差地别。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
import heapq

def single_threaded_order(tasks):
    pending = sorted(((start, duration, i) for i, (start, duration) in enumerate(tasks)))
    ready = []
    i = 0
    now = 0
    out = []
    while i < len(pending) or ready:
        if not ready:
            now = max(now, pending[i][0])
        while i < len(pending) and pending[i][0] <= now:
            start, duration, index = pending[i]
            heapq.heappush(ready, (duration, index))
            i += 1
        duration, index = heapq.heappop(ready)
        out.append(index)
        now += duration
    return out

def assign_tasks(servers, tasks):
    if not servers and tasks:
        raise ValueError('at least one server required')
    available = [(weight, i) for i, weight in enumerate(servers)]
    heapq.heapify(available)
    busy = []
    now = 0
    out = []
    for arrival, duration in enumerate(tasks):
        now = max(now, arrival)
        if not available and busy:
            now = max(now, busy[0][0])
        while busy and busy[0][0] <= now:
            _, weight, index = heapq.heappop(busy)
            heapq.heappush(available, (weight, index))
        weight, index = heapq.heappop(available)
        out.append(index)
        heapq.heappush(busy, (now + duration, weight, index))
    return out

# 示例与回归测试
assert single_threaded_order([[1, 2], [2, 4], [3, 2], [4, 1]]) == [0, 2, 3, 1]

assert single_threaded_order([[100, 2], [1000, 1]]) == [0, 1]

assert assign_tasks([3, 3, 2], [1, 2, 3, 2, 1, 2]) == [2, 2, 0, 2, 1, 2]

from random import Random

rng = Random(61)

for _ in range(200):
    weights = [rng.randrange(1, 5) for _ in range(rng.randrange(1, 5))]
    tasks = [rng.randrange(1, 8) for _ in range(15)]
    free = [0] * len(weights)
    now = 0
    ref = []
    for arrival, duration in enumerate(tasks):
        now = max(now, arrival)
        if all((t > now for t in free)):
            now = min(free)
        i = min((i for i, t in enumerate(free) if t <= now), key=lambda i: (weights[i], i))
        ref.append(i)
        free[i] = now + duration
    assert assign_tasks(weights, tasks) == ref

print('N061: 所有本章断言通过')

N061: 所有本章断言通过


## 11. 代表题与迁移

- **1834. Single-Threaded CPU（单线程 CPU）**：`single_threaded_order` 的原题，练的是"就绪堆复合键 (时长,下标) + 空闲跳转"这套事件驱动写法。
- **1882. Process Tasks Using Servers（使用服务器处理任务）**：`assign_tasks` 的原题，练的是"available 与 busy 双堆各按各的键排序、到期搬运"的调度骨架。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。